# 🏥 New Medical QLoRA Fine-Tuning
### Saves directly to Google Drive — No more lost files!
> ⚠️ **First:** Set Runtime to T4 GPU → Runtime → Change runtime type → T4 GPU


## ⚡ Step 1 — Keep Colab Alive (Run This First!)


In [ ]:
# This prevents Colab from disconnecting
from IPython.display import Javascript
display(Javascript("""
function keepAlive() {
    document.querySelector("#top-toolbar").click();
    console.log("Keeping Colab alive!");
}
setInterval(keepAlive, 30000);
"""))
print("✅ Keep alive started!")


## 💾 Step 2 — Connect Google Drive (Click Allow On Everything!)


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)

# Create folder in Google Drive
DRIVE_SAVE_PATH = "/content/drive/MyDrive/medical_lora_adapter"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

print("✅ Google Drive connected!")
print(f"✅ Save folder ready: {DRIVE_SAVE_PATH}")


## 📦 Step 3 — Install Libraries (Wait 3-5 Minutes)


In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout[-200:] if result.stdout else "Done")

print("Installing Unsloth...")
run('pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')

print("Installing other libraries...")
run("pip install --quiet datasets trl peft accelerate bitsandbytes transformers")

print("\n✅ All installed!")


## 🖥️ Step 4 — Check GPU


In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {gpu.name}")
    print(f"✅ VRAM: {gpu.total_memory/1e9:.1f} GB")
else:
    print("❌ No GPU! Go to Runtime → Change runtime type → T4 GPU")


## 🤖 Step 5 — Load Model (Wait 2-4 Minutes)


In [ ]:
from unsloth import FastLanguageModel

# Settings
MODEL_NAME  = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"  # Small fast model
MAX_SEQ_LEN = 1024

print(f"Loading {MODEL_NAME}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

print("\n✅ Model loaded!")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")


## 🔧 Step 6 — Add LoRA Adapters


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA ready!")
print(f"   Trainable: {trainable:,} ({100*trainable/total:.2f}%)")


## 📚 Step 7 — Load Medical Dataset


In [ ]:
from datasets import load_dataset

print("Loading medical dataset...")

try:
    ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
    ds = ds.rename_columns({"input": "question", "output": "answer"})
except:
    ds = load_dataset("medalpaca/medical_meadow_health_advice", split="train")

# Use only 500 samples for fast training (~5 minutes)
ds = ds.shuffle(seed=42).select(range(500))

SYSTEM_PROMPT = "You are MedAssist, an expert medical AI assistant. Provide accurate and helpful medical information."

def format_example(example):
    question = example.get("question") or example.get("input") or example.get("instruction", "")
    answer   = example.get("answer")   or example.get("output") or example.get("response", "")
    if not question or not answer:
        return {"text": ""}
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{question.strip()}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{answer.strip()}<|eot_id|>"
    )
    return {"text": text}

dataset = ds.map(format_example, remove_columns=ds.column_names)
dataset = dataset.filter(lambda x: len(x["text"]) > 50)

print(f"✅ Dataset ready: {len(dataset):,} samples")


## 🚀 Step 8 — Train! (Wait ~5-10 Minutes)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import time

print("Starting training...")
print("Files saving to Google Drive automatically!")

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    args               = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio                = 0.05,
        num_train_epochs            = 1,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        output_dir                  = "/content/drive/MyDrive/medical_checkpoints",
        save_steps                  = 50,
        save_total_limit            = 1,
        report_to                   = "none",
    ),
)

start = time.time()
result = trainer.train()
elapsed = time.time() - start

print(f"\n✅ Training complete!")
print(f"   Time      : {elapsed/60:.1f} minutes")
print(f"   Final loss: {result.training_loss:.4f}")


## 💾 Step 9 — Save Adapter To Google Drive


In [ ]:
import os

DRIVE_SAVE_PATH = "/content/drive/MyDrive/medical_lora_adapter"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

print("Saving to Google Drive...")
model.save_pretrained(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)

files = os.listdir(DRIVE_SAVE_PATH)
size  = sum(os.path.getsize(os.path.join(DRIVE_SAVE_PATH, f)) for f in files) / 1e6

print(f"\n✅ Saved to Google Drive!")
print(f"   Path  : {DRIVE_SAVE_PATH}")
print(f"   Files : {files}")
print(f"   Size  : {size:.1f} MB")
print("\n🎉 Go to drive.google.com and download medical_lora_adapter folder!")


## 🧪 Step 10 — Test Your Medical AI!


In [ ]:
FastLanguageModel.for_inference(model)

def ask_medical(question):
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{question}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with __import__("torch").inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 300,
            temperature        = 0.3,
            do_sample          = True,
            repetition_penalty = 1.15,
            pad_token_id       = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Test questions
questions = [
    "What are symptoms of diabetes?",
    "How is high blood pressure treated?",
    "What causes a heart attack?",
]

for q in questions:
    print(f"\nQuestion: {q}")
    print("-" * 50)
    print(ask_medical(q))

print("\n✅ All done! Your medical AI is working!")


## 💬 Step 11 — Ask Your Own Question!


In [ ]:
# Change this to any medical question you want!
MY_QUESTION = "What is the difference between Type 1 and Type 2 Diabetes?"

print(f"Question: {MY_QUESTION}")
print("=" * 50)
print(ask_medical(MY_QUESTION))


## 🔄 Step 12 — Convert to GGUF & Connect to Dr. Deerfly UI!
> This converts your fine-tuned adapter into GGUF format so you can load it into **Ollama** and use it with the **Dr. Deerfly UI**!
> ⏱️ Takes about 5-10 minutes. Make sure Google Drive is still mounted.

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 12 — Convert to GGUF for Ollama + Dr. Deerfly UI
# ═══════════════════════════════════════════════════════════

import os
from google.colab import files

GGUF_DIR = "/content/medical_gguf"
DRIVE_GGUF = "/content/drive/MyDrive/medical_gguf"

print("Step 1/3 — Merging LoRA adapter into base model...")
print("This takes 3-5 minutes...")

try:
    # Merge adapter weights into full model
    model.save_pretrained_merged(
        "/content/medical_merged",
        tokenizer,
        save_method="merged_16bit"
    )
    print("✅ Step 1 done — Model merged!")

    print("
Step 2/3 — Converting to GGUF format (q4_k_m)...")
    print("This takes 3-5 minutes...")

    # Convert to GGUF — this is what Ollama needs
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method="q4_k_m"
    )
    print("✅ Step 2 done — GGUF file created!")

    # Also save a copy to Google Drive so you dont lose it
    os.makedirs(DRIVE_GGUF, exist_ok=True)
    os.system(f"cp {GGUF_DIR}/*.gguf {DRIVE_GGUF}/")
    print("✅ GGUF also backed up to Google Drive!")

    # List the file
    gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
    print(f"
📁 GGUF file created: {gguf_files}")

    print("
Step 3/3 — Downloading GGUF file to your PC...")
    print("⚠️  File is 2-4 GB — may take a few minutes to download!")

    # Download to PC
    for f in gguf_files:
        files.download(f"{GGUF_DIR}/{f}")
        print(f"✅ Downloading: {f}")

except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure model is still loaded — try running from Cell 10 first")

print("""
═══════════════════════════════════════════════════
NEXT STEPS ON YOUR PC:
═══════════════════════════════════════════════════
1. Wait for download to finish
2. Create a file called Modelfile (no extension) with:

   FROM ./unsloth.Q4_K_M.gguf
   SYSTEM """You are MedAssist, a medical AI trained on clinical data."""
   PARAMETER temperature 0.7
   PARAMETER num_ctx 2048

3. Open CMD in same folder as the .gguf file
4. Run: ollama create medassist -f Modelfile
5. Run: ollama run medassist
6. Open Dr. Deerfly UI and select medassist from dropdown!
═══════════════════════════════════════════════════
""")
